# CAPI Quality-in-Use Evaluation — Reliability and User Satisfaction

This notebook analyzes the questionnaire responses collected to answer **RQ3**:

> **How do real users of CAPI perceive its overall software quality, as measured across the nine quality aspects drawn from the ISO/IEC 25010 Quality in Use and Product Quality models?**

The notebook reports:

- **Instrument reliability** using **Cronbach's alpha** for each quality aspect and for the overall questionnaire.
- **Per-aspect satisfaction level**, computed as the mean score of each ISO/IEC 25010 quality aspect.
- **Overall satisfaction mean**, computed across all questionnaire items.
- Descriptive statistics for each questionnaire item.
- Reliability interpretation following George and Mallery (2003).

**Measurement decisions**:
- Questionnaire responses are measured on a **five-point Likert scale** (1 = Strongly disagree, ..., 5 = Strongly agree).
- Six quality aspects are measured using two questionnaire items, while three aspects are measured using a single item.
- Cronbach's alpha is reported only for aspects with at least two items; for single-item aspects, internal consistency is undefined.
- Satisfaction levels are calculated as arithmetic means and interpreted according to the predefined satisfaction categories.

Requires: `pandas`, `numpy`, `scipy`, `openpyxl`

In [ ]:

from pathlib import Path
import pandas as pd

# Repo root, resolved relative to this file — works regardless of where you run it from.
ROOT = Path(__file__).resolve().parents[1]   # notebook: Path.cwd().parents[0]
DATA = ROOT / "03_Data"

r1 = pd.read_csv(DATA / "rater2_ratings.csv")
r2 = pd.read_csv(DATA / "rater1_ratings.csv")

df = pd.read_csv(file_path)
df.columns=[str(c).replace('\xa0','').strip() for c in df.columns]
print(df.columns.tolist())

ITEMS={
'Effectiveness':['EFF1'],
'Efficiency':['EFI1','EFI2'],
'Satisfaction':['SAT1','SAT2'],
'Freedom from Risk':['FFR1'],
'Reliability':['REL1','REL2'],
'Security':['SEC1','SEC2'],
'Context Coverage':['CTX1','CTX2'],
'Learnability':['LRN1'],
'Accessibility':['ACC1', 'ACC1.1']
}

def cronbach_alpha(d):
    d=d.dropna()
    k=d.shape[1]
    if k<2:
        return np.nan
    v=d.var(ddof=1,axis=0)
    t=d.sum(axis=1).var(ddof=1)
    return (k/(k-1))*(1-v.sum()/t)

print("\nPer-dimension Cronbach's alpha")
for a,cols in ITEMS.items():
    miss=[c for c in cols if c not in df.columns]
    if miss:
        print(a, "missing:", miss)
        continue
    if len(cols)==1:
        print(a,": single item (alpha undefined)")
    else:
        print(a,":",round(cronbach_alpha(df[cols]),3))

multi=[c for cols in ITEMS.values() if len(cols)>1 for c in cols]
print("\nOverall alpha:", round(cronbach_alpha(df[multi]),3))


### Satisfaction Level Per Item

In [ ]:
print("\nSatisfaction Level Per Item:")
for category, cols in ITEMS.items():
    # Filter out columns that are not in the DataFrame or are single items which are undefined for satisfaction level
    valid_cols = [c for c in cols if c in df.columns]

    if not valid_cols:
        print(f"{category}: No valid columns found in DataFrame.")
        continue

    if len(valid_cols) == 1:
        # For single items, the satisfaction level is simply the mean of that column
        mean_value = df[valid_cols[0]].mean()
        print(f"{category} ({valid_cols[0]}): {mean_value:.4f}")
    elif len(valid_cols) > 1:
        # For multiple items in a category, calculate the mean of the means for each column
        means = df[valid_cols].mean()
        overall_mean = means.mean()
        print(f"{category}: {overall_mean:.3f}")
    else:
        print(f"{category}: Not enough items to calculate satisfaction level.")

In [ ]:
print("\nOverall Score:")

# Flatten the ITEMS dictionary to get all relevant columns
all_items_cols = []
for category, cols in ITEMS.items():
    all_items_cols.extend(cols)

# Filter out columns that are not in the DataFrame
valid_all_items_cols = [c for c in all_items_cols if c in df.columns]

if not valid_all_items_cols:
    print("No valid columns found in DataFrame to calculate overall score.")
else:
    # Calculate the mean of all valid item columns
    overall_score = df[valid_all_items_cols].mean().mean()
    print(f"Overall Score: {overall_score:.4f}")